This file is generated for the data cleaning process regarding this sub-question (5).

In [1]:
# Importing libraries
import pandas as pd
import numpy as np
import os
from scipy.optimize import curve_fit

Loading merged csv file as a dataframe

In [ ]:
# Defining full file path
file_path = r"C:\Users\Ecem\Desktop\TIL\1. TIL6022 - Python\Project\TIL6022-Group-Project\Data\Processed\merged_eurostat_clean_V2.csv"

# Loading dataset
df = pd.read_csv(file_path)

Filtering necessary columns and rows

In [15]:
eu_countries = [
    "AT","BE","BG","HR","CY","CZ","DK","EE","FI","FR","DE","EL","GR","HU","IE",
    "IT","LV","LT","LU","MT","NL","PL","PT","RO","SK","SI","ES","SE"
]

# Filtering the dataset to include only EU-27 countries
df = df[df["geo"].isin(eu_countries)].copy()

# Selecting relevant columns
base_features = [
    "geo", 
    "TIME_PERIOD",
    "Consignment_full_train_load_THS_T",
    "Consignment_full_wagon_load_THS_T",
    "Consignment_total_THS_T",
]

# Detecting all commodity columns dynamically (they start with "NST_GT")
commodity_columns = [col for col in df.columns if col.startswith("NST_GT")]

# Combining base and commodity columns
selected_columns = base_features + commodity_columns

# Subset the dataframe
df_subset = df[selected_columns].copy()

# Sort by country and year before interpolation
df_subset = df_subset.sort_values(by=["geo", "TIME_PERIOD"]).reset_index(drop=True)

Data completeness check

In [16]:
consignment_cols = [
    "Consignment_full_train_load_THS_T",
    "Consignment_full_wagon_load_THS_T",
    "Consignment_total_THS_T"
]

# Computing missing data percentage per country
missing_by_country = (
    df_subset.groupby("geo")[consignment_cols]
    .apply(lambda g: g.isna().mean() * 100)
    .round(1)
    .sort_values(by="Consignment_full_train_load_THS_T", ascending=False)
)

print("Missing ratio (%) per country (for consignment columns)")
print(missing_by_country)

Missing ratio (%) per country (for consignment columns)
     Consignment_full_train_load_THS_T  Consignment_full_wagon_load_THS_T  \
geo                                                                         
AT                               100.0                              100.0   
ES                               100.0                              100.0   
PT                               100.0                              100.0   
NL                               100.0                              100.0   
LV                               100.0                              100.0   
LU                               100.0                              100.0   
BE                               100.0                              100.0   
FR                               100.0                              100.0   
HU                               100.0                              100.0   
EL                               100.0                              100.0   
EE                  

In [17]:
# Computing number of missing years per country (for filtering)
missing_years_per_country = (
    df_subset.groupby("geo")["Consignment_full_train_load_THS_T"]
    .apply(lambda x: x.isna().sum())
    .sort_values(ascending=False)
)

print("Number of missing years per country (train consignment)")
print(missing_years_per_country)

Number of missing years per country (train consignment)
geo
AT    17
ES    17
PT    17
NL    17
LV    17
LU    17
FR    17
HU    17
EL    17
CZ    17
BG    17
DK    17
EE    17
BE    16
DE     7
HR     3
FI     2
IE     1
SE     1
IT     0
LT     0
PL     0
RO     0
SI     0
SK     0
Name: Consignment_full_train_load_THS_T, dtype: int64


In [18]:
# Defining a threshold for exclusion
# If a country has more than 10 years of missing consignment data, exclude it.
threshold_missing_years = 10

countries_to_keep = missing_years_per_country[
    missing_years_per_country <= threshold_missing_years
].index.tolist()

countries_removed = missing_years_per_country[
    missing_years_per_country > threshold_missing_years
].index.tolist()

print(f"Countries kept  : {len(countries_to_keep)} → {countries_to_keep}")
print(f"Countries removed: {len(countries_removed)} → {countries_removed}")

Countries kept  : 11 → ['DE', 'HR', 'FI', 'IE', 'SE', 'IT', 'LT', 'PL', 'RO', 'SI', 'SK']
Countries removed: 14 → ['AT', 'ES', 'PT', 'NL', 'LV', 'LU', 'FR', 'HU', 'EL', 'CZ', 'BG', 'DK', 'EE', 'BE']


In [19]:
# Filtering dataset to keep only eligible countries
df_subset = df_subset[df_subset["geo"].isin(countries_to_keep)].copy()

print(f"Final dataset after filtering countries: {df_subset['geo'].nunique()} countries")
print(f"Rows remaining: {len(df_subset)}")

Final dataset after filtering countries: 11 countries
Rows remaining: 187


Interpolation to fill missing data

In [22]:
def fill_with_curvefit(df, col_name, time_col="TIME_PERIOD", group_col="geo"):

    df_filled = df.copy().sort_values(by=[group_col, time_col])
    
    def linear_func(x, a, b):
        return a * x + b

    interpolated_groups = []

    for country, group in df_filled.groupby(group_col):
        y = group[col_name].values
        x = group[time_col].values

        valid_mask = ~np.isnan(y)

        # Skipping interpolation when data is insufficient
        if valid_mask.sum() < 2:
            interpolated_groups.append(group)
            continue

        x_valid = x[valid_mask]
        y_valid = y[valid_mask]
        x_centered = x_valid - x_valid.mean()
        x_all_centered = x - x_valid.mean()

        try:
            popt, _ = curve_fit(linear_func, x_centered, y_valid)
            y_pred = linear_func(x_all_centered, *popt)
            group.loc[np.isnan(y), col_name] = y_pred[np.isnan(y)]
        except Exception as e:
            print(f"Skipped {country} for {col_name} due to fitting error: {e}")

        interpolated_groups.append(group)

    return pd.concat(interpolated_groups, ignore_index=True)

In [23]:
# Applying interpolation to train and wagon consignment columns
df_subset = fill_with_curvefit(df_subset, "Consignment_full_train_load_THS_T")
df_subset = fill_with_curvefit(df_subset, "Consignment_full_wagon_load_THS_T")

# Recalculating total
df_subset["Consignment_total_THS_T"] = (
    df_subset["Consignment_full_train_load_THS_T"] +
    df_subset["Consignment_full_wagon_load_THS_T"]
)

Converting the table to analysis-ready format

In [27]:
# Melting commodity columns into long format
df_long = df_subset.melt(
    id_vars=[
        "geo", "TIME_PERIOD",
        "Consignment_full_train_load_THS_T",
        "Consignment_full_wagon_load_THS_T",
        "Consignment_total_THS_T"
    ],
    value_vars=[col for col in df_subset.columns if col.startswith("NST_GT")],
    var_name="NST2007",
    value_name="Volume_MIO_TKM"
)

# Cleaning commodity codes
df_long["NST2007"] = (
    df_long["NST2007"]
    .str.replace("NST_GT", "")
    .str.replace("_MIO_TKM", "")
)

# Mapping NST codes to readable names
nst_labels = {
    "01": "Agriculture and forestry products",
    "02": "Coal and crude petroleum",
    "03": "Metal ores and quarrying products",
    "04": "Food, beverages, tobacco",
    "05": "Textiles and leather products",
    "06": "Wood and cork products",
    "07": "Paper and printed matter",
    "08": "Refined petroleum products",
    "09": "Chemicals and man-made fibres",
    "10": "Rubber and plastic products",
    "11": "Non-metallic mineral products",
    "12": "Basic and fabricated metals",
    "13": "Machinery and equipment",
    "14": "Electrical machinery",
    "15": "Transport equipment",
    "16": "Furniture and other goods",
    "17": "Secondary raw materials and waste",
    "18": "Grouped goods (mixed consignments)",
    "19": "Unidentifiable goods",
    "20": "Empty packaging and return loads"
}

df_long["NST2007_desc"] = df_long["NST2007"].map(nst_labels)

# Dropping numeric code column
df_long = df_long.drop(columns=["NST2007"])

# Computing consignment shares
df_long["train_share"] = np.where(
    df_long["Consignment_total_THS_T"] > 0,
    df_long["Consignment_full_train_load_THS_T"] / df_long["Consignment_total_THS_T"],
    np.nan
)

df_long["wagon_share"] = np.where(
    df_long["Consignment_total_THS_T"] > 0,
    df_long["Consignment_full_wagon_load_THS_T"] / df_long["Consignment_total_THS_T"],
    np.nan
)

# Reordering columns for clarity
df_clean_final = df_long[
    [
        "geo", "TIME_PERIOD", "NST2007_desc", "Volume_MIO_TKM",
        "Consignment_full_train_load_THS_T",
        "Consignment_full_wagon_load_THS_T",
        "Consignment_total_THS_T",
        "train_share", "wagon_share"
    ]
].copy()

df_clean_final.to_csv(r"C:\Users\Ecem\Desktop\TIL\1. TIL6022 - Python\Project\df_clean_final_subQ5.csv", index=False)

# Display the entire DataFrame
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

display(df_clean_final)

,geo,TIME_PERIOD,NST2007_desc,Volume_MIO_TKM,Consignment_full_train_load_THS_T,Consignment_full_wagon_load_THS_T,Consignment_total_THS_T,train_share,wagon_share
0,DE,2008,Agriculture and forestry products,3799.0,267134.000000,104164.000000,371298.000000,0.719460,0.280540
1,DE,2009,Agriculture and forestry products,3320.0,265869.850209,100361.315958,366231.166168,0.725962,0.274038
2,DE,2010,Agriculture and forestry products,2783.0,262955.000000,92984.000000,355939.000000,0.738764,0.261236
3,DE,2011,Agriculture and forestry products,2190.0,269220.000000,105581.000000,374801.000000,0.718301,0.281699
4,DE,2012,Agriculture and forestry products,1661.0,270644.000000,95560.000000,366204.000000,0.739053,0.260947
5,DE,2013,Agriculture and forestry products,1490.0,267938.000000,105337.000000,373275.000000,0.717803,0.282197
6,DE,2014,Agriculture and forestry products,1486.0,257216.000000,107810.000000,365026.000000,0.704651,0.295349
7,DE,2015,Agriculture and forestry products,1473.0,261849.287448,106037.296219,367886.583667,0.711766,0.288234
8,DE,2016,Agriculture and forestry products,1552.0,243435.000000,106359.000000,349794.000000,0.695938,0.304062
9,DE,2017,Agriculture and forestry products,1547.0,240100.000000,124033.000000,364133.000000,0.659374,0.340626
